# Transfer-Kernel Ranging (TKR) — Full Pipeline & Paper Figures

This notebook is the reference implementation of **Transfer-Kernel Ranging (TKR)**, the
point-source methane localisation and quantification method described in:

> Klappenbach et al., *"Novel method to locate and quantify point-source methane emissions
> using time series of ground-based column observations"*, EGUsphere, 2026.
> https://doi.org/10.5194/egusphere-2026-204

It is a single, self-contained notebook: **Part 1** generates all data (observations, peak
detection, trajectory loading, upwind-domain segmentation, transport-kernel fitting) for
every available peak. **Part 2** builds Figures 1-6 of the paper from that data. All
non-trivial logic lives in [`tkr_functions.py`](./tkr_functions.py); this notebook mainly
wires those functions together and produces plots.

> **Project naming note:** the method was developed under the working title "Local Source
> Projection" (LSP); the released method/paper name is **Transfer-Kernel Ranging (TKR)**.
> File and variable names in this repository use `TKR`/`tkr_*` throughout.

**Runtime note:** Part 1 processes every peak with locally available trajectory data
(~130 s/peak in testing) — expect ~15–30 minutes total depending on peak count and machine.
Progress is printed per peak.

**Required inputs** (paths configured in `config.json`, see also the repository README):
- `df_dens.parquet` — aggregated trajectory/site metadata
- `<traj_dir>/<time>-<site_id>-total-column/about.json` + `traj/particle_stilt.*.parquet` per peak
- the EM27/SUN retrieval-bundle parquet (observations)
- the averaging-kernel JSON

None of these are included in this Git repository (the full dataset is ~8 GB) — see the
README for the download link and the expected folder layout under `demo_data/`.

# Part 1 — Generate data

## 1.1 Setup: imports, config, base data

In [ ]:
import os
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from scipy.signal import convolve
from pvlib.solarposition import get_solarposition

from tkr_functions import (
    # geometry / atmosphere helpers
    haversine_distance, calculate_initial_bearing, circular_mean,
    pressure_std_atm, molec_column, load_ak,
    # trajectory loading
    calculate_wind_vectors, interpolate_particle_trajectories, read_traj_parq,
    # upwind-domain segmentation
    create_radius, bearing_segmentation, ring_area,
    # Step 1 + 2: kernel fit & emission-strength inversion
    make_histogram,
    # observations & peak finding
    read_obs_proffast_parquet, find_group_peaks, find_nearest,
    # run I/O, wind-speed correction, unit conversion, candidate matching, KMZ export
    save_results, load_results, dual_time_formatter_factory,
    apply_wind_speed_correction, convert_emissions, match_candidate,
    build_kmz,
)

warnings.filterwarnings('ignore')

with open('config.json') as f:
    CONFIG = json.load(f)

# `obs_location_id` and `utc_offset_hours` are not (yet) part of config.json but are
# needed by `read_obs_proffast_parquet`; set them here so everything stays visible in
# one place. See the README for what `location_id` means and why it can differ from
# `site_id` for the same physical site.
CONFIG.setdefault('obs_location_id', None)
CONFIG.setdefault('utc_offset_hours', -7)  # PDT; only used for the local calendar-day boundary

site_dw = CONFIG['site_dw']  # short site key, e.g. used in some input filenames (ma_avk_*.json)
site_id = CONFIG['site_id']  # site id used in the exported trajectory folder names

df_dens = pd.read_parquet(CONFIG['df_dens_path'])

print(f"df_dens: {df_dens.shape}")

## 1.2 Observations and peak detection (paper Sect. 2.1, Appendix B)

In [ ]:
observations = read_obs_proffast_parquet(
    CONFIG['observations_path'], date=CONFIG['date'], species=CONFIG['target_gas'].upper(),
    quantile=CONFIG['quantile'], roll_time=CONFIG['roll_time'],
    location_id=CONFIG['obs_location_id'], utc_offset_hours=CONFIG['utc_offset_hours'],
)
observations = observations.reset_index().set_index('minutes')

# Detect enhancement peaks within each gap-free chunk of the day (paper Fig. 1).
peak_data = observations.groupby('nan_chunk_index').apply(
    lambda x: find_group_peaks(x, key='Enh_ppm', prominence=CONFIG['prominence']), include_groups=False
)

# Snap each observed peak time onto the discrete time grid on which trajectories were
# pre-computed, so peaks can be matched to an exported trajectory folder later.
minutes_grid = np.unique(df_dens.minutes)
peak_data['peak_time_grid'] = [find_nearest(minutes_grid, x) for x in peak_data.peak_time]
peak_data = peak_data.reset_index().set_index('minutes')

print(f"observations: {observations.shape}, peaks found: {len(peak_data)}")

## 1.3 Averaging-kernel weighting (paper Appendix D)

In [ ]:
# Receptor location and altitude (site metadata, config.json).
la_0 = CONFIG['receptor_lat']
lo_0 = CONFIG['receptor_lon']
z_instr = CONFIG['receptor_alt_asl_m']  # instrument altitude above sea level (m)

m_air = 28.97 / 1000  # kg/mol, molar mass of dry air

# Vertical layer thickness (m) around each trajectory release altitude, from the
# midpoints between adjacent altitude levels in df_dens.
alts = np.unique(df_dens.alt)
diffs = np.diff(np.concatenate(([-alts[0]], alts)))
layers = [0.5 * (diffs[i + 1] + diffs[i]) for i in range(len(alts) - 1)]
layers.append(layers[-1])
layer_fun = {alts[i]: layers[i] for i in range(len(alts))}
df_dens['layer_thickness'] = df_dens['alt'].apply(lambda x: layer_fun[x])

# Solar zenith angle at each trajectory release time -> needed for the averaging
# kernel A(sza, pressure) (paper Appendix D).
df_dens['utc'] = df_dens['recep'].apply(str).apply(lambda x: pd.to_datetime(x, format='%Y%m%d%H%M', utc=True))
df_dens['sza'] = 90 - get_solarposition(
    df_dens['utc'].to_numpy(), la_0, lo_0, altitude=None, pressure=None,
    method='nrel_numpy', temperature=12.0
)['apparent_elevation'].to_numpy()

# Pressure at each release altitude (barometric formula; altitudes in df_dens are
# given above ground level, so add the instrument's altitude above sea level).
pressures = pressure_std_atm(df_dens['alt'] + z_instr)
ak = load_ak(CONFIG['averaging_kernel_path'])
df_dens['ak'] = ak((df_dens['sza'], pressures))

# Per-trajectory averaging-kernel weight (paper Appendix D, Eq. D1): fraction of the
# total dry-air column represented by this altitude layer, times the averaging kernel.
df_dens['layer_molec_m2'] = df_dens['density'] / m_air * df_dens['layer_thickness'] / 1000
df_dens['dens_layer_weight'] = df_dens['layer_molec_m2'] / molec_column(z_instr)
df_dens['full_weight_per_trajectory'] = df_dens['dens_layer_weight'] / df_dens['numpar'] * df_dens['ak']

print('Averaging-kernel weighting complete.')

## 1.4 Which peak to inspect in detail

Figures 2 and 4 need one peak's full segment-level detail (including the fitted time series
for the inset plots). Figure 5 and the multi-peak residual validation use **all** peaks.
Set `SELECTED_PEAKTIME_GRID` below (or leave it as the first available peak).

In [ ]:
def available_peaktime_grids(config, peak_data):
    '''Peaks for which a trajectory folder was actually exported to disk.'''
    out = []
    for ptg in np.unique(peak_data.peak_time_grid):
        ctime = pd.to_datetime(config['date']) + pd.to_timedelta('%imin' % int(ptg))
        p = f"{config['traj_dir']}/{ctime.strftime('%Y%m%d-%H%M')}-{config['site_id']}-total-column/about.json"
        if os.path.exists(p):
            out.append((ptg, ctime))
    return out

PEAK_OPTIONS = available_peaktime_grids(CONFIG, peak_data)
print(f"{len(PEAK_OPTIONS)} peak(s) with local trajectory data:")
for ptg, ctime in PEAK_OPTIONS:
    row = peak_data[peak_data.peak_time_grid == ptg].iloc[0]
    print(f"  peaktime_grid={ptg:>7.1f}  ({ctime.strftime('%H:%M')} UTC)  peak height: {row.peak_height*1000:.1f} ppb")

# --- SELECT PEAK HERE ---
SELECTED_PEAKTIME_GRID = PEAK_OPTIONS[0][0]  # e.g. set explicitly: SELECTED_PEAKTIME_GRID = 990.0
# ------------------------
print(f"\nSelected for detailed Figures 2/4: peaktime_grid={SELECTED_PEAKTIME_GRID}")

## 1.5 Process every available peak (Step 1 & 2, paper Sect. 2.5-2.6, Eq. 5-8)

For each peak: load trajectories, segment the upwind domain into (altitude, distance,
bearing) bins, and fit the transport kernel per segment. The **selected** peak's full
result (including per-segment time series, needed for the Fig. 2/4 insets) is kept in
`trajectories` / `fitted_to_obs`. A slimmed-down summary for **every** peak is collected
in `fitted_by_peak` / `all_fits`, used by Figure 5, Figure 6, and the multi-peak residual
validation.

This is the slow step (see the runtime note in the notebook header).

In [ ]:
# Radial bin edges (Appendix E) and their corresponding angular bin width (Appendix F).
radius_steps_km, delta_alpha_deg = create_radius(
    CONFIG['drf'], dr=CONFIG['dr'], rad_max=CONFIG['rad_max_km'], max_steps=CONFIG['max_radial_steps'], d_alpha=True
)
source_altitude_segmentation = np.linspace(0, CONFIG['max_source_altitude_m'], CONFIG['source_altitude_bins'])

fitted_by_peak = {}   # per-peak fit summary, ALL processed peaks (used by Fig. 5/6 + validation)
trajectories = None   # full segmented trajectories, kept only for the selected peak (Fig. 2)
fitted_to_obs = None  # full per-segment fit results (incl. time series), selected peak only (Fig. 2/4)

t_start = time.time()
for n, (peaktime_grid, ctime) in enumerate(PEAK_OPTIONS):
    t0 = time.time()
    traj_path = f"{CONFIG['traj_dir']}/{ctime.strftime('%Y%m%d-%H%M')}-{site_id}-total-column/"

    # --- Load this peak's backward trajectories (paper Sect. 2.3) ---
    traj_column, meta = read_traj_parq(
        traj_path, select_release_heights=range(CONFIG['release_heights']),
        cutoff_time=CONFIG['trajectory_cutoff_minutes'], hi_res_time=CONFIG['hi_res_time_resolution'],
        averaging_kernel_path=CONFIG['averaging_kernel_path'],
    )

    # --- Segment the upwind domain into (altitude, distance, bearing) bins (Sect. 2.3.1, Appendix E) ---
    cur_traj = traj_column.copy().reset_index(drop=True)
    cur_traj = cur_traj[cur_traj.recep_dist_km < 10].reset_index(drop=True)
    cur_traj['recep_dist_segmentation'] = pd.cut(cur_traj['recep_dist_km'], radius_steps_km, right=False)
    cur_traj['source_agl_segmentaion'] = pd.cut(cur_traj.zagl, source_altitude_segmentation, right=False)
    intervals = pd.IntervalIndex.from_breaks(source_altitude_segmentation, closed='left')
    dz_dict = {interval: interval.right - interval.left for interval in intervals}
    cur_traj['dz_source'] = cur_traj['source_agl_segmentaion'].map(dz_dict)
    cur_traj['dalpha'] = np.interp(cur_traj['recep_dist_km'], radius_steps_km,
                                    np.concatenate([delta_alpha_deg[1:], [delta_alpha_deg[-1]]]))
    a = cur_traj.groupby(['source_agl_segmentaion', 'recep_dist_segmentation'], observed=True).apply(
        bearing_segmentation, include_groups=True
    )
    a_flat = a.reset_index(level=[0, 1], drop=True)
    cur_traj = cur_traj.join(a_flat[['recep_bearing_segmentation', 'deg_step']], how='inner')
    cur_traj['segment_area_m2'] = ring_area(cur_traj['recep_dist_segmentation'], cur_traj['recep_bearing_segmentation'])

    # --- Observed enhancement window around this peak ---
    peak_row = peak_data[peak_data.peak_time_grid == peaktime_grid].iloc[0]
    peak_time = peak_row.peak_time
    pw = CONFIG['peak_window_minutes']
    peak_section = observations[(observations.index <= peak_time + pw / 2) & (observations.index > peak_time - pw / 2)]

    # --- Step 1 + 2: fit the transport kernel per segment (Eq. 5-8) ---
    cur_fitted = cur_traj.groupby(
        ['source_agl_segmentaion', 'recep_dist_segmentation', 'recep_bearing_segmentation'], observed=True
    ).apply(
        lambda x: make_histogram(
            x, peak_section, peak_time, std_dev=2, weight=False, plot=False,
            max_plot_radius=0.3, debug=False, emission_duration=True,
            hi_res_time_resolution=CONFIG['hi_res_time_resolution'],
        ),
        include_groups=False,
    ).reset_index()

    cur_fitted['recep_dist_km'] = cur_fitted['recep_dist_segmentation'].apply(lambda x: 0.5 * (x.left + x.right)).astype(float)
    cur_fitted['recep_bearing_deg_mid'] = cur_fitted['recep_bearing_segmentation'].apply(
        lambda x: 0.5 * (x.left + x.right) if pd.notna(x) else np.nan).astype(float)
    cur_fitted['duration_residual_std_ppm'] = cur_fitted['duration_residual'].apply(lambda x: np.nanstd(x))
    cur_fitted['peak_time'] = peak_time
    cur_fitted['peaktime_grid'] = peaktime_grid

    fitted_by_peak[peaktime_grid] = cur_fitted.copy()

    if peaktime_grid == SELECTED_PEAKTIME_GRID:
        trajectories = cur_traj
        fitted_to_obs = cur_fitted.copy()

    dt = time.time() - t0
    print(f"[{n+1}/{len(PEAK_OPTIONS)}] {ctime.strftime('%H:%M')} UTC done in {dt:.0f}s "
          f"(total elapsed {(time.time()-t_start)/60:.1f} min), {len(cur_fitted)} segments")

all_fits = pd.concat(fitted_by_peak.values(), ignore_index=True)
print(f"\nDONE. Total time: {(time.time()-t_start)/60:.1f} min. all_fits shape: {all_fits.shape}")

## 1.6 Save / reload results

`save_results` / `load_results` (in `tkr_functions.py`) pickle everything Part 1 produced,
so Part 2 (figure generation) can be re-run or edited without repeating the slow trajectory
processing above.

In [ ]:
save_results(
    filepath=f"{CONFIG['output_dir']}/fit_results_{site_id}.pkl",
    all_fits=all_fits,
    fitted_by_peak=fitted_by_peak,
    trajectories=trajectories,
    fitted_to_obs=fitted_to_obs,
    radius_steps_km=radius_steps_km,
    delta_alpha_deg=delta_alpha_deg,
    source_altitude_segmentation=source_altitude_segmentation,
    selected_peaktime_grid=SELECTED_PEAKTIME_GRID,
    config=CONFIG,
)

In [ ]:
results = load_results(f"{CONFIG['output_dir']}/fit_results_{site_id}.pkl")

all_fits = results['all_fits']
fitted_by_peak = results['fitted_by_peak']
trajectories = results['trajectories']
fitted_to_obs = results['fitted_to_obs']
radius_steps_km = results['radius_steps_km']
delta_alpha_deg = results['delta_alpha_deg']
source_altitude_segmentation = results['source_altitude_segmentation']
SELECTED_PEAKTIME_GRID = results['selected_peaktime_grid']

del results

# Part 2 — Paper figures

## Figure 1: observed enhancement and detected peaks

In [ ]:
UTC_OFFSET_HOURS = CONFIG['utc_offset_hours']
base = pd.to_datetime(CONFIG['date'])
obs_time = base + pd.to_timedelta(observations.index, unit='m')
peak_time_dt = base + pd.to_timedelta(peak_data.peak_time, unit='m')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(obs_time, observations.Enh_ppm * 1000, lw=0.7, label='observations', color='tab:blue')
ax.plot(peak_time_dt, peak_data.peak_height * 1000, 'ro', label='peak', ms=4)

# Reference 12-minute grid: the source's suspected periodicity (paper Fig. 1 caption).
line_starts = np.arange(peak_data.peak_time.min(), peak_data.peak_time.max(), 12)
for x in line_starts:
    ax.axvline(base + pd.to_timedelta(x, unit='m'), lw=0.2, color='k')
ax.axvline(base + pd.to_timedelta(line_starts[-1], unit='m'), lw=0.2, color='k', label='12 min interval')

ax.xaxis.set_major_formatter(FuncFormatter(dual_time_formatter_factory(UTC_OFFSET_HOURS)))
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
ax.set_xlim(base + pd.Timedelta('16:15:00'), base + pd.Timedelta('19:30:00'))
ax.set_xlabel('UTC (local time)')
ax.set_ylabel('Enhancement XCH4 / ppb')
ax.legend()
plt.tight_layout()
plt.savefig('figures/Figure1.png', dpi=250)
print('saved figures/Figure1.png')

## Figure 1 (supplement): PROFFAST 2.2 vs. 2.4.1 retrieval comparison

Reprocessing consistency check mentioned in Sect. 2.1: compares peak amplitudes retrieved
with PROFFAST 2.2 (used throughout the paper) against a reprocessing with PROFFAST 2.4.1.
Requires a second retrieval-bundle parquet with `-2_2-` in place of `-2_4-` in the filename
(same naming convention as `observations_path` in `config.json`); skip this cell if you only
have one retrieval version available.

In [ ]:
observations_prf_2_2 = read_obs_proffast_parquet(
    CONFIG['observations_path'].replace('-2_4-', '-2_2-'), date=CONFIG['date'], species=CONFIG['target_gas'].upper(),
    quantile=CONFIG['quantile'], roll_time=CONFIG['roll_time'],
    location_id=CONFIG['obs_location_id'], utc_offset_hours=CONFIG['utc_offset_hours'],
)
observations_prf_2_2 = observations_prf_2_2.reset_index().set_index('minutes')

peak_data_2_2 = observations_prf_2_2.groupby('nan_chunk_index').apply(
    lambda x: find_group_peaks(x, key='Enh_ppm', prominence=CONFIG['prominence']), include_groups=False
)
peak_data_2_2['peak_time_grid'] = [find_nearest(minutes_grid, x) for x in peak_data_2_2.peak_time]
peak_data_2_2 = peak_data_2_2.reset_index().set_index('minutes')

print(f"observations: {observations_prf_2_2.shape}, peaks found: {len(peak_data_2_2)}")

# Restrict both retrievals to their common time range/grid before comparing.
common_min = max(observations_prf_2_2.index.min(), observations.index.min())
common_max = min(observations_prf_2_2.index.max(), observations.index.max())
obs_2_2 = observations_prf_2_2.loc[common_min:common_max].sort_index()
obs_2_4 = observations.loc[common_min:common_max].sort_index()
common_idx = obs_2_2.index.intersection(obs_2_4.index)
obs_2_2 = obs_2_2.loc[common_idx]
obs_2_4 = obs_2_4.loc[common_idx]
diff_enh = (obs_2_4['Enh_ppm'] - obs_2_2['Enh_ppm']) * 1000  # ppb

time_2_2 = base + pd.to_timedelta(observations_prf_2_2.index, unit='m')
time_2_4 = base + pd.to_timedelta(observations.index, unit='m')
time_diff = base + pd.to_timedelta(common_idx, unit='m')
peak_time_2_2 = base + pd.to_timedelta(peak_data_2_2.peak_time, unit='m')
peak_time_2_4 = base + pd.to_timedelta(peak_data.peak_time, unit='m')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(time_2_2, observations_prf_2_2.Enh_ppm * 1000, lw=0.7, label='prf 2.2', color='tab:orange')
ax1.plot(time_2_4, observations.Enh_ppm * 1000, lw=0.7, label='prf 2.4.1', color='tab:blue')
ax1.plot(peak_time_2_2, peak_data_2_2.peak_height * 1000, 'o', color='tab:orange', ms=4, label='peak 2.2')
ax1.plot(peak_time_2_4, peak_data.peak_height * 1000, 'o', color='tab:blue', ms=4, label='peak 2.4.1')
ax1.set_ylabel('Enhancement XCH4 / ppb')
ax1.legend()

ax2.plot(time_diff, diff_enh, lw=0.7, color='tab:green')
ax2.axhline(0, lw=0.5, color='k')
ax2.set_ylabel('\u0394 Enh. (2.4.1 \u2212 2.2) / ppb')
ax2.xaxis.set_major_formatter(FuncFormatter(dual_time_formatter_factory(UTC_OFFSET_HOURS)))
ax2.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
ax2.set_xlim(base + pd.Timedelta('16:15:00'), base + pd.Timedelta('19:30:00'))
ax2.set_xlabel('UTC (local time)')

plt.tight_layout()
plt.savefig('figures/Figure1_difference.png', dpi=250)
print('saved figures/Figure1_difference.png')

# Relative change in peak amplitude for the 13 forenoon peaks (paper Sect. 2.1: -2.9%
# median, up to -6.5%; an order of magnitude below the transport-driven uncertainty).
n_forenoon = 13
change = (peak_data.peak_height.to_numpy()[:n_forenoon] - peak_data_2_2.peak_height.to_numpy()[:n_forenoon]) / peak_data.peak_height.to_numpy()[:n_forenoon]
print('max change: %1.3f %%' % (np.max(np.abs(change)) * 100))
print('median change: %1.3f %%' % (np.median(change) * 100))

## Figure 2: particle segmentation illustration

Built from the **selected peak** (`trajectories` / `fitted_to_obs` from Part 1). `ALT_LAYER_INDEX`
picks which single altitude layer is shown in the map; colors there distinguish
distance-bearing segments *within* that layer only (see the paper's Fig. 2 caption), not
altitude. The three example segments (near/mid/far, well-sampled) are chosen automatically.

In [ ]:
ALT_LAYER_INDEX = 8
MAX_RAD_KM = 3

alt_bins = sorted(trajectories['source_agl_segmentaion'].cat.categories)
alt = alt_bins[ALT_LAYER_INDEX]
cdf = trajectories[(trajectories.source_agl_segmentaion == alt) & (trajectories.recep_dist_km < MAX_RAD_KM)].copy()
cfit = fitted_to_obs[(fitted_to_obs.source_agl_segmentaion == alt) & (fitted_to_obs.recep_dist_km < MAX_RAD_KM)].copy()
cfit = cfit[cfit['hist_points'] > 0].set_index(['recep_dist_segmentation', 'recep_bearing_segmentation'])

print(f"Altitude layer: {alt} ({len(cdf)} particle-timesteps, {len(cfit)} fitted segments)")

In [ ]:
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

fig = plt.figure(figsize=(16 / 1.2, 9 / 1.2))
gs = gridspec.GridSpec(3, 2, width_ratios=[1, 3])
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax3 = fig.add_subplot(gs[2, 0], sharex=ax1)
for ax in [ax1, ax2]:
    plt.setp(ax.get_xticklabels(), visible=False)
ax_big = fig.add_subplot(gs[:, 1])
ax_big.set_aspect(1)

# Scatter every particle position (in this altitude layer), one color per (distance,
# bearing) segment, in the receptor-centered east/north plane (paper Fig. 2, right panel).
colors = {}
for name, group in cdf.groupby(['recep_dist_segmentation', 'recep_bearing_segmentation'], observed=True):
    sc = ax_big.scatter(group.recep_dist_east_km, group.recep_dist_north_km, s=3)
    if name in cfit.index:
        colors[name] = sc.get_facecolor()

ax_big.set_xlabel('receptor distance east / km')
ax_big.set_ylabel('receptor distance north / km')
ax_big.yaxis.set_label_position('right')
ax_big.yaxis.tick_right()

to_ppb = 1000
cdf_idx = cdf.set_index(['recep_dist_segmentation', 'recep_bearing_segmentation'])

# Pick three example segments (near / mid / far upwind distance) among the well-sampled
# ones, to illustrate how the transport kernel broadens with distance (Fig. 2, left panel).
valid_names = [n for n in colors.keys() if cfit.loc[n, 'hist_points'] > 100]
valid_sorted = sorted(valid_names, key=lambda n: cdf_idx.loc[n, 'recep_dist_km'].mean()
                       if hasattr(cdf_idx.loc[n, 'recep_dist_km'], 'mean') else cdf_idx.loc[n, 'recep_dist_km'])
picks = [valid_sorted[3], valid_sorted[int(np.floor(len(valid_sorted) / 1.5))], valid_sorted[-2]]

axes = [ax1, ax2, ax3]
time_center = None
points_ax, points_big = [], []

for ax, cname in zip(axes, picks):
    cf = cfit.loc[cname]
    upwind_dist = cdf_idx.loc[cname].recep_dist_km.mean()
    c = colors[cname]
    if time_center is None:
        time_center = cf.time_minutes[np.argmax(cf.observations_ppm)]
    if ax == ax1:
        ax.plot((cf.time_minutes - time_center) * 60, cf.kernel_fitted * to_ppb, lw=6, color=c, label='kernel %1.1fm' % (upwind_dist*1000))
    else:
        ax.plot((cf.time_minutes - time_center) * 60, cf.kernel_fitted * to_ppb, lw=6, color=c, label='kernel %1.1fkm' % upwind_dist)
    ax.plot((cf.time_minutes - time_center) * 60, cf.observations_ppm * to_ppb, linestyle=':', color='k', label='observed')
    points_ax.append((((cf.time_minutes - time_center) * 60)[np.argmax(cf.kernel_fitted)], np.max(cf.kernel_fitted) * to_ppb))
    pt_big = (cdf_idx.loc[cname].recep_dist_east_km.mean(), cdf_idx.loc[cname].recep_dist_north_km.mean())
    points_big.append(pt_big)
    ax_big.scatter(*pt_big, c='k')
    ax.set_ylabel('enhancement / ppb')
    ax.legend(loc=2)
ax3.set_xlabel('time / s')

def trans_fig(ax, x, y):
    '''Map (x, y) data coordinates of `ax` into figure-fraction coordinates, so an
    arrow can be drawn between two different subplots (used to point from each small
    kernel-fit inset to its segment in the big map).'''
    bbox = ax.get_position()
    xlm = ax.get_xlim()
    xout = (bbox.x1 - bbox.x0) / (xlm[1] - xlm[0]) * (x - xlm[0]) + bbox.x0
    ylm = ax.get_ylim()
    yout = (bbox.y1 - bbox.y0) / (ylm[1] - ylm[0]) * (y - ylm[0]) + bbox.y0
    return (xout, yout)

plt.tight_layout()
for i, ax in enumerate(axes):
    start = trans_fig(ax, *points_ax[i])
    end = trans_fig(ax_big, *points_big[i])
    arrow = FancyArrowPatch(start, end, transform=fig.transFigure, color='k', arrowstyle='-|>',
                             mutation_scale=15, lw=0.8, connectionstyle='angle,angleA=0,angleB=90,rad=0')
    fig.add_artist(arrow)

plt.savefig('figures/Figure2.png', dpi=250)
print('saved figures/Figure2.png')

## Figure 2 (supplement): candidate source locations in geographic coordinates

Same particle cloud as Figure 2, but plotted in lon/lat instead of receptor-relative
east/north, with the candidate source locations overlaid (defined once in
`SOURCE_LOCATIONS` and reused by Figures 5 and 6 below).

In [ ]:
fig = plt.figure(figsize=(16 / 1.2, 9 / 1.2))
ax_big = plt.gca()
ax_big.set_aspect(1)

colors = {}
for name, group in cdf.groupby(['recep_dist_segmentation', 'recep_bearing_segmentation'], observed=True):
    sc = ax_big.scatter(group.long, group.lati, s=3)
    if name in cfit.index:
        colors[name] = sc.get_facecolor()

ax_big.set_xlabel('longitude')
ax_big.set_ylabel('latitude')
ax_big.yaxis.set_label_position('right')
ax_big.yaxis.tick_right()

# Candidate source locations investigated in the source search (paper Sect. 3, Table 1).
# Defined once here and reused by Figures 5 and 6.
SOURCE_LOCATIONS = {
    'foothill dining':   (37.87544915758012, -122.25616344191205, 'potential candidate, very close'),
    'student housing':   (37.87608233036625, -122.25645436619163, 'potential candidate, very close, broken heating system reported'),
    'NG infrastructure': (37.87556109517469, -122.25443161396028, 'natural gas valve, surrounded by trees, injection into atmosphere not likely'),
    'building 30':       (37.87649436515283, -122.24708452824339, 'smoke stack approximately 25m above ground'),
    'building 33':       (37.87609216845105, -122.24674281410584, 'smoke stack approximately 40m above ground'),
    'tank':              (37.87846958360055, -122.24143278375334, 'unknown tank, potential large source, injection into atmosphere possible due to height'),
    'LBNL1':             (37.877097985303465, -122.24861278713782, 'location of meteorological site, no source candidate'),
}

for key in SOURCE_LOCATIONS.keys():
    ax_big.scatter(SOURCE_LOCATIONS[key][1], SOURCE_LOCATIONS[key][0], label=key)

ax_big.scatter(lo_0, la_0, s=30, c='r', label='receptor')
ax_big.legend(fontsize=7, loc='upper left')
plt.tight_layout()
print('candidate locations plotted (not saved to disk; diagnostic view only)')

## Figure 3: emission / kernel / convolution illustration

Synthetic, independent of the generated data — illustrates the LTI convolution
relationship (paper Appendix A, Eq. 1) and how the same observed enhancement can be
explained by different (kernel, emission) pairs depending on assumed source distance.

In [ ]:
t = np.arange(0, 900, 1)
kt = np.arange(-300, 301, 1)

# The "observed" signal to be reproduced: a narrow synthetic peak.
measured_signal = np.exp(-0.5 * ((t - 450) / 20) ** 2)
measured_signal /= measured_signal.max()

def narrow_kernel(t):
    '''Kernel for a source close to the receptor (narrow spread in arrival time).'''
    k = np.exp(-0.5 * (t / 10) ** 2)
    return k / k.sum()

def wide_kernel(t):
    '''Kernel for a source further upwind (broader spread, Sect. 2.5).'''
    k = np.exp(-0.5 * (t / 50) ** 2)
    return k / k.sum()

def very_wide_kernel(t):
    '''Kernel for a source far enough upwind that even a puff emission can't
    reproduce the narrow observed peak (the 'beyond r_max' case, Sect. 2.5).'''
    k = np.exp(-0.5 * (t / 100) ** 2)
    return k / k.sum()

kernels = [narrow_kernel(kt), wide_kernel(kt), very_wide_kernel(kt)]

# Row 1: a long-duration emission through a narrow kernel reproduces the peak.
em1 = np.exp(-0.5 * ((t - 450) / 50) ** 2)
em1 *= measured_signal.max() / convolve(em1, kernels[0], mode='same').max()
# Row 2: at larger distance, a shorter emission is needed to reproduce the same peak.
em2 = np.exp(-0.5 * ((t - 450) / 10) ** 2)
em2 *= measured_signal.max() / convolve(em2, kernels[1], mode='same').max()
# Row 3: even an instantaneous (delta-function) emission through the very wide kernel
# cannot reproduce the observed peak shape -- this distance is beyond r_max.
em3 = np.zeros_like(t)
em3[450] = 1.0 / 0.004000045722115894

emissions = [em1, em2, em3]
convolutions = [convolve(e, k, mode='same') for e, k in zip(emissions, kernels)]

fig, axes = plt.subplots(3, 3, figsize=(15 / 1.5, 9 / 1.5), sharex='col')
column_titles = ['emission\n[\u00b5mol/m\u00b2/s]', 'kernel\n[1/(mol/m\u00b2/s)]', 'convolution\n[\u00b5mol/mol]']
for ax, title in zip(axes[0], column_titles):
    ax.set_title(title)

for i in range(3):
    axes[i][0].plot(t-450, emissions[i], label='emission', color='tab:blue')
    axes[i][0].set_xlim(-300, 300)
    axes[i][1].plot(kt, kernels[i], label='kernel', color='tab:orange')
    axes[i][1].set_xlim(-300, 300)
    axes[i][2].plot(t-450, convolutions[i], label='conv', color='tab:red')
    axes[i][2].plot(t-450, convolutions[0], label='observ.', linestyle=':', color='black')
    axes[i][2].set_xlim(-300, 300)
    for j in range(3):
        if i == 2:
            axes[i][j].set_xlabel('t / s')
        axes[i][j].grid(True)

axes[0][2].legend(loc=1)
plt.tight_layout()
plt.subplots_adjust(wspace=0.4)
plt.savefig('figures/Figure3.png', dpi=200)
print('saved figures/Figure3.png')

## Figure 4: residual vs. distance, with inset and emission bar chart (paper Sect. 2.5-2.6)

Built from the **selected peak** (`fitted_to_obs` from Part 1). The top panel's minimum
identifies `r_max` (Eq. 6); the inset compares the observation to the best-fitting kernel
(Step 1) and kernel*emission (Step 2) reconstructions; the bottom panel shows the
corresponding retrieved emission (Eq. 8) as a function of assumed upwind distance.

In [ ]:
def plot_figure4(fitted_to_obs, hist_points_min=500, out_path='figures/Figure4.png'):
    '''Reproduce paper Figure 4 for one peak's full per-segment fit results.

    Aggregates the per-segment fit residual (Step 1, Eq. 6) and retrieved emission
    (Step 2, Eq. 8) by radial distance bin (averaging over altitude/bearing segments
    at that distance), finds r_max as the distance of minimum residual, and picks one
    representative segment closest to r_max for the inset time-series comparison.
    '''
    fitted_to_obs = fitted_to_obs.copy()
    well_sampled = fitted_to_obs[fitted_to_obs['hist_points'] > hist_points_min].copy()
    if len(well_sampled) == 0:
        raise ValueError(f"No segments with hist_points > {hist_points_min}.")

    cols = ['recep_dist_km', 'residual_std_ppm', 'duration_residual_std_ppm', 'emission_mol', 'hist_points']
    me_df = well_sampled.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanmean)
    st_df = well_sampled.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanstd)
    r_max_km = me_df.loc[me_df['residual_std_ppm'].idxmin(), 'recep_dist_km']

    well_sampled['dist_to_rmax'] = (well_sampled['recep_dist_km'] - r_max_km).abs()
    example_row = well_sampled.sort_values('dist_to_rmax').iloc[0]

    fig, (a0, a1) = plt.subplots(2, 1, sharex=True, figsize=(8, 5.5), gridspec_kw={'height_ratios': [3, 1]})
    a0.errorbar(me_df['recep_dist_km'], me_df['duration_residual_std_ppm'] * 1000, yerr=st_df['duration_residual_std_ppm'] * 1000,
                label='kernel * emissions', fmt='-o', lw=0.5, capsize=3, c='red')
    a0.errorbar(me_df['recep_dist_km'], me_df['residual_std_ppm'] * 1000, yerr=st_df['residual_std_ppm'] * 1000,
                fmt=':d', label='kernel only', capsize=3, lw=0.5, c='orange')
    a0.axvline(r_max_km, color='r', lw=0.8, ls='--', label=f'r_max = {r_max_km:.2f} km')
    a0.set_ylabel('fit residual std / ppb')

    peak_time = well_sampled.peak_time.iloc[0]
    ctime = base + pd.Timedelta(minutes=float(peak_time))
    local_ctime = base + pd.Timedelta(minutes=(float(peak_time) + CONFIG['utc_offset_hours'] * 60))
    a0.set_title(f"{ctime:%H:%M} UTC" + f"  ({local_ctime:%H:%M} local)")

    a01 = a0.inset_axes([0.5, 0.0, 0.5, 0.5])
    t_ = example_row['time_minutes']
    a01.plot(t_, example_row['observations_ppm'] + example_row['duration_residual'], c='red', label='kernel * emission', zorder=9)
    a01.plot(t_, example_row['kernel_fitted'], c='orange', label='kernel', zorder=10)
    a01.plot(t_, example_row['observations_ppm'], linestyle=':', color='black', zorder=11, label='observation')
    a01.get_xaxis().set_visible(False)
    a01.get_yaxis().set_visible(False)
    a01.legend(fontsize=6, loc='upper right')
    a0.legend(loc='upper center')

    wdh = np.diff(me_df['recep_dist_km']) * 0.8
    wdh = np.concatenate((wdh, [wdh[-1]]))
    a1.bar(me_df['recep_dist_km'], me_df['emission_mol'], yerr=st_df['emission_mol'], width=wdh)
    a1.set_yscale('log')
    a1.set_ylabel('total emission / mol')
    a1.set_xlabel('upwind distance / km')
    plt.tight_layout()
    plt.savefig(out_path, dpi=250)
    print(f"r_max = {r_max_km:.3f} km, well-sampled segments: {len(well_sampled)} / {len(fitted_to_obs)}")
    return fig, r_max_km

_ = plot_figure4(fitted_to_obs, out_path='figures/Figure4.png')

## Figure 5: candidate source emission estimates (paper Table 1 / Fig. 5)

Uses `all_fits` from Part 1 — all peaks processed in this notebook run, no external file
needed beyond the wind-speed correction data below. For each candidate location, finds the
matching upwind segment for every available peak (`match_candidate`, geometric distance +
bearing match) and computes the per-peak, per-candidate emission estimate. A candidate can
be unmatched for a given peak if that peak's wind direction didn't sample the candidate's
bearing.

**Wind-speed correction (paper Sect. 2.7.2, Appendix H):** the paper's Fig. 5 uses the
observed LBNL1 wind speed in place of the HRRR model wind speed (Eq. 9); this needs a local
meteorological station CSV that is **not** part of the published demo dataset (see the
README). If you don't have it, skip the `apply_wind_speed_correction` cell and work with
`result` (uncorrected) instead of `result_corr`.

In [ ]:
def match_all_peaks(all_fits, source_locations, la_0, lo_0, hist_points_min=500):
    '''Run `match_candidate` for every (candidate location, peak) pair, returning one
    row per successful match with the retrieved emission summary statistics.'''
    met_agl = np.unique(all_fits['source_agl_segmentaion'])[2]
    records = []
    for name, (lat, lon, comment) in source_locations.items():
        for peak_time_val, peak_group in all_fits.groupby('peak_time'):
            matched, d_km, bearing = match_candidate(peak_group, lat, lon, la_0, lo_0, hist_points_min=hist_points_min)
            if len(matched) == 0:
                continue
            records.append({
                'candidate': name, 'peak_time': peak_time_val,
                'emission_median': matched['emission_mol'].median(),
                'emission_min': matched['emission_mol'].min(),
                'emission_max': matched['emission_mol'].max(),
                'emission_duration_min': matched['emission_duration_min'].mean(),
                'dist_km': d_km, 'bearing_deg': bearing, 'n_segments': len(matched),
                'wind_speed': matched[matched['source_agl_segmentaion'] == met_agl]['wind_speed'].mean(),
                'wind_speed_std': matched[matched['source_agl_segmentaion'] == met_agl]['wind_speed'].std(),
                'wind_dir_deg': matched[matched['source_agl_segmentaion'] == met_agl]['wind_dir_deg'].mean(),  # not a circular mean
                'wind_dir_deg_std': matched[matched['source_agl_segmentaion'] == met_agl]['wind_dir_deg'].std(),  # not a circular std
            })
    return pd.DataFrame.from_records(records)

fig5_data = match_all_peaks(all_fits, SOURCE_LOCATIONS, la_0, lo_0)
print(f"{len(fig5_data)} (candidate, peak) matches across {fig5_data['candidate'].nunique()} candidates "
      f"and {fig5_data['peak_time'].nunique()} of {all_fits['peak_time'].nunique()} processed peaks."
      if len(fig5_data) else "No matches found.")
fig5_data.sort_values(['peak_time', 'candidate'])

In [ ]:
# mol/peak -> g/s, kg/peak, t CH4/yr (paper Table 1 units).
result = convert_emissions(fig5_data)

# Wind-speed correction against the local LBNL1 meteorological station (paper Appendix H).
# Skip this cell (and use `result` instead of `result_corr` below) if you don't have the
# station CSV -- see the README for details.
result_corr = apply_wind_speed_correction(
    result,
    met_path=f"{CONFIG['data_dir']}/LBNL1_{CONFIG['date']}.csv",
    reference_candidate='LBNL1',
)

unit_cols = [c for c in result_corr.columns if c.endswith(('_g_s_wcorr', '_kg_peak_wcorr', '_tCH4_yr_wcorr'))]
summary = result_corr.groupby('candidate')[unit_cols].agg(['mean', 'min', 'max'])
summary

### Table 1 as LaTeX

In [ ]:
def summary_to_latex(summary, cols=None, caption="CH4 emissions per candidate", label="tab:ch4_emissions"):
    '''Format the `summary` DataFrame (output of `result_corr.groupby('candidate')[unit_cols].agg(['mean','min','max'])`)
    as a LaTeX table matching the style of the paper's Table 1.'''
    df = summary.copy()

    if cols is None:
        cols = [c for c in df.columns.get_level_values(0).unique()]

    label_map = {
        'emission_median_g_s': 'g/s',
        'emission_median_kg_peak': 'kg/peak',
        'emission_median_tCH4_yr': 'tCH4/yr',
        'emission_min_g_s': 'g/s (min)',
        'emission_min_kg_peak': 'kg/peak (min)',
        'emission_min_tCH4_yr': 'tCH4/yr (min)',
        'emission_max_g_s': 'g/s (max)',
        'emission_max_kg_peak': 'kg/peak (max)',
        'emission_max_tCH4_yr': 'tCH4/yr (max)',
    }

    flat = pd.DataFrame(index=df.index)
    for col in cols:
        for stat in ['mean', 'min', 'max']:
            new_name = f"{label_map.get(col, col)} ({stat})"
            flat[new_name] = df[(col, stat)]

    flat = flat.reset_index()

    return flat.to_latex(
        index=False,
        float_format="%.2f",
        caption=caption,
        label=label,
        column_format='l' + 'r' * (len(flat.columns) - 1),
        escape=True,
    )

print(summary_to_latex(summary, cols=['emission_median_g_s_wcorr', 'emission_median_kg_peak_wcorr', 'emission_median_tCH4_yr_wcorr']))

### Figure 5 plot

In [ ]:
fig5_plot_data = result_corr.copy()

fig, ax = plt.subplots(figsize=(11, 5))

peak_times = sorted(fig5_plot_data['peak_time'].unique())
candidates = list(SOURCE_LOCATIONS.keys())
n_cand = len(candidates)
bar_width = 0.8 / n_cand
colors = plt.cm.tab10(np.linspace(0, 1, n_cand))

for i, cand in enumerate(candidates):
    if cand in ['LBNL1', 'tank']:  # not real emission candidates (see paper Sect. 3)
        continue

    sub = fig5_plot_data[fig5_plot_data['candidate'] == cand].set_index('peak_time').reindex(peak_times)
    x = np.arange(len(peak_times)) + (i - n_cand / 2 + 0.5) * bar_width
    yerr = np.vstack([
        (sub['emission_median_wcorr'] - sub['emission_min_wcorr']).clip(lower=0).fillna(0),
        (sub['emission_max_wcorr'] - sub['emission_median_wcorr']).clip(lower=0).fillna(0),
    ])
    if len(sub.dist_km.dropna()) == 0:
        continue
    ax.bar(x, sub['emission_median_wcorr'].fillna(0), width=bar_width, color=colors[i],
           label=cand + ' (%1.0fm)' % (sub.dist_km.dropna().iloc[0] * 1000))
    ax.errorbar(x, sub['emission_median_wcorr'], yerr=yerr, fmt='none', ecolor='k', elinewidth=0.7, capsize=2)

ax.set_yscale('log', base=10)
ax.set_ylim((10, 4e5))
ax.set_xticks(np.arange(len(peak_times)))
xticklabels = [(base + pd.Timedelta(minutes=pt)).strftime('%H:%M') +
               (base + pd.Timedelta(minutes=pt + UTC_OFFSET_HOURS * 60)).strftime('\n(%H:%M)') for pt in peak_times]
ax.set_xticklabels(xticklabels, rotation=90, ha='right')

ax.set_xlabel('peak time UTC (local)')
ax.set_ylabel('CH4 peak emission / mol')
ax.legend(fontsize=8, ncol=2)
ax.set_title('peak emissions per candidate')

ax2 = ax.twinx()
ax2.set_yscale('log', base=10)
ax2.set_ylim(np.array(ax.get_ylim()) * 16.04 / 1000)
ax2.set_ylabel('CH4 peak emission / kg')

plt.tight_layout()
plt.savefig('figures/Figure5.png', dpi=250)
print('saved figures/Figure5.png')

## Multi-peak validation figure (paper Appendix G, Fig. G1)

Reproduces the residual-vs-distance curve of Figure 4 for **every** processed peak in one
grid, to check that the r_max minimum is well-defined across peaks (paper Appendix G: 12 of
13 forenoon peaks show a single, clear minimum; one, at 16:38 UTC, shows the flatter,
less-constrained minimum expected under the "multiple wind pathways" worst case).

In [ ]:
import matplotlib.lines as mlines

HIST_POINTS_MIN = 500
peak_groups = sorted(all_fits['peaktime_grid'].unique())
n_peaks = len(peak_groups)
ncols, nrows = 2, 7

fig, axes = plt.subplots(nrows, ncols, figsize=(8.27, 11.69), sharex=True, sharey=True)  # A4 portrait
axes_flat = axes.flatten()

cols = ['recep_dist_km', 'residual_std_ppm', 'duration_residual_std_ppm']

for i, ptg in enumerate(peak_groups):
    ax = axes_flat[i]
    sub = fitted_by_peak[ptg]
    peak_time = sub.peak_time.iloc[0]
    well = sub[sub['hist_points'] > HIST_POINTS_MIN].copy()
    ctime = base + pd.Timedelta(minutes=float(peak_time))

    if len(well) == 0:
        ax.text(0.5, 0.5, 'no well-sampled\nsegments', ha='center', va='center', fontsize=7, transform=ax.transAxes)
        ax.set_title(f"{ctime:%H:%M} UTC", fontsize=8)
        continue

    me = well.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanmean)
    st = well.groupby('recep_dist_segmentation', observed=True)[cols].agg(np.nanstd)
    r_max = me.loc[me['residual_std_ppm'].idxmin(), 'recep_dist_km']

    well['dist_to_rmax'] = (well['recep_dist_km'] - r_max).abs()
    example_row = well.sort_values('dist_to_rmax').iloc[0]

    ax.errorbar(me['recep_dist_km'], me['duration_residual_std_ppm'] * 1000, yerr=st['duration_residual_std_ppm'] * 1000,
                fmt='-o', c='red', ms=2.5, lw=0.5, capsize=1.5, elinewidth=0.5)
    ax.errorbar(me['recep_dist_km'], me['residual_std_ppm'] * 1000, yerr=st['residual_std_ppm'] * 1000,
                fmt=':d', c='orange', ms=2.5, lw=0.5, capsize=1.5, elinewidth=0.5)
    ax.axvline(r_max, color='r', lw=0.6, ls='--')
    ax.set_title(f"{ctime:%H:%M} UTC, r$_{{max}}$={r_max:.2f} km", fontsize=8)
    ax.tick_params(labelsize=7)

    ax_inset = ax.inset_axes([0.5, 0, 0.5, 0.5])
    t_ = example_row['time_minutes']
    ax_inset.plot(t_, example_row['observations_ppm'] + example_row['duration_residual'], c='red', lw=0.6, zorder=9)
    ax_inset.plot(t_, example_row['kernel_fitted'], c='orange', lw=0.6, zorder=10)
    ax_inset.plot(t_, example_row['observations_ppm'], linestyle=':', color='black', lw=0.6, zorder=11)
    ax_inset.get_xaxis().set_visible(False)
    ax_inset.get_yaxis().set_visible(False)

for j in range(n_peaks, nrows * ncols):
    axes_flat[j].axis('off')

if n_peaks < nrows * ncols:
    leg_ax = axes_flat[n_peaks]
    handles = [
        mlines.Line2D([], [], color='orange', ls=':', marker='d', ms=4, lw=0.8, label='kernel only'),
        mlines.Line2D([], [], color='red', ls='-', marker='o', ms=4, lw=0.8, label='kernel * emissions'),
        mlines.Line2D([], [], color='r', ls='--', lw=0.8, label=r'$r_{max}$'),
        mlines.Line2D([], [], color='black', ls=':', lw=0.8, label='observation (inset)'),
    ]
    leg_ax.legend(handles=handles, loc='center', fontsize=9, frameon=False)

fig.supxlabel('upwind distance / km', fontsize=9)
fig.supylabel('fit residual std / ppb', fontsize=9)
plt.tight_layout(rect=[0.02, 0.02, 1, 1])
plt.savefig('figures/Figure_multi_peak_validation.png', dpi=250)
print(f"saved figures/Figure_multi_peak_validation.png ({n_peaks} peaks)")

## Figure 6: Google Earth (KMZ) export of the fit-residual point cloud

Reproduces paper Figure 6: for each peak, exports a `.kmz` file with the upwind segments
colored by normalised fit residual (cyan = low residual / likely source region, magenta =
high residual), plus labelled markers for the receptor and the candidate source locations.
Open the resulting files in Google Earth (Pro or web). Requires the `simplekml` package
(`pip install simplekml`).

In [ ]:
CANDIDATES_TO_PLOT = [
    'foothill dining',
    'student housing',
    'NG infrastructure',
    'building 30',
    'building 33',
    'LBNL1',
]  # 'tank' excluded: not carried forward as a candidate in this revision (see paper Sect. 3)

build_kmz(
    fitted_by_peak, base, lo_0, la_0,
    source_locations=SOURCE_LOCATIONS,
    candidates_to_plot=CANDIDATES_TO_PLOT,
    hist_points_min=HIST_POINTS_MIN,
    out_dir='./kmz_out',
)